In [ ]:
import run_judge
from importlib import reload

In [ ]:
reload(run_judge)
run_judge.main()

In [ ]:
import pandas as pd
pd.read_csv('/home/jmayer/ACI_research/Code/resources/judge_results.csv')

In [ ]:
import judge_msi
import pandas as pd
import os
import sys
from run_judge import trim_to_stcu_region
from importlib import reload
sys.path.append('./../../../ectools/')
from ectools import ecio

In [ ]:
import glob
orbit = '06311E'
region = 'SEA'
month = '07'
day = '08'
year = '2025'

MCOP_filename = f'/home/jmayer/Object_Data/MSI_COP_2A/{year}/{month}/{day}/*{orbit}.h5'
MRGR_filename = f'/home/jmayer/Object_Data/MSI_RGR_1C/{year}/{month}/{day}/*{orbit}.h5'
ACMCAP_filename = f'/home/jmayer/Object_Data/ACM_CAP_2B/{year}/{month}/{day}/*{orbit}.h5'

In [ ]:
import glob
orbit = '05286E'
region = 'NEA'
month = '05'
day = '03'
year = '2025'

MCOP_filename = f'/home/jmayer/Object_Data/MSI_COP_2A/{year}/{month}/{day}/*{orbit}.h5'
MRGR_filename = f'/home/jmayer/Object_Data/MSI_RGR_1C/{year}/{month}/{day}/*{orbit}.h5'
ACMCAP_filename = f'/home/jmayer/Object_Data/ACM_CAP_2B/{year}/{month}/{day}/*{orbit}.h5'

In [ ]:
MCOP_file = glob.glob(MCOP_filename)[0]
MRGR_file = glob.glob(MRGR_filename)[0]
ACMCAP_file = glob.glob(ACMCAP_filename)[0]

MCOP = ecio.load_MCOP(MCOP_file)
MRGR = ecio.load_MRGR(MRGR_file)
MCOP = trim_to_stcu_region(MCOP, region, orbit[-1])
MRGR = trim_to_stcu_region(MRGR, region, orbit[-1])
MRGR = MRGR.where(MRGR.TIR1 < 1e30)

In [ ]:
MRGR

In [ ]:
reload(judge.mytorchutils)
print('Running judge...')
cats, cert, scene_midpoints = judge.judge_MSI(MCOP, MRGR, debugging=False)
print('\tDone!')

In [ ]:
scene_midpoints

In [ ]:
df = pd.DataFrame(scene_midpoints)
df['cats'] = cats
df['cert'] = cert
df['orbit'] = orbit

In [ ]:
from run_judge import plot_judge_results
plot_judge_results(MCOP, MRGR, df, orbit, region, year, month, day, save_to=None)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_context('talk', font_scale=0.8)
fig, axs = plt.subplots(2, 1, figsize=(35,6), sharex=True)
plt.suptitle(
    f'Orbit {orbit}, Region {region}, Date 2025-{month}-{day}, '
    f'lat: {np.nanmax(MCOP.latitude.values[0,:]):.2f} to {np.nanmax(MCOP.latitude.values[-1,:]):.2f}, '
    f'lon: {np.nanmax(MCOP.longitude.values[0,:]):.2f} to {np.nanmax(MCOP.longitude.values[-1,:]):.2f}',
    fontsize = 20
)
MCOP.cloud_optical_thickness.T.plot(ax=axs[0], vmin=1, vmax=50)
cbar = axs[0].collections[0].colorbar
cbar.set_label('COT', fontsize=20)

MRGR.TIR2.T.plot(ax=axs[1], vmin=273, vmax=300)
cbar = axs[1].collections[0].colorbar
cbar.set_label('TIR2 (K)', fontsize=20)

hue = np.where(df.ice_flag == 1, 6, df.cats)
colors = ['#1f77b4', '#2ca02c', '#ff7f0e',  '#d62728', '#e377c2', '#8c564b', 'white'] # last color is for ice
palette = {0: colors[0], 1: colors[1], 2: colors[2], 3: colors[3], 4: colors[4], 5: colors[5], 6: colors[6]}
for i in [0,1]:
    legend = True if i == 1 else False
    scatter = sns.scatterplot(ax=axs[i], x = df.along_track_mid, y=MRGR.dims['across_track'] - 128, hue=hue, size=df.cert, size_norm=(0.3,1),
                    marker='o', palette=palette, legend=legend)
    if legend and scatter.legend_ is not None:
        scatter.legend_.set_title('')
fig.tight_layout()

In [ ]:
scene_midpoints['cloud_cover']

In [ ]:
stratus_regions = {
    'SEA': {'lat': (-25, -5), 'lon': (-5, 15)},    # South East Atlantic
    'NEA': {'lat': (10, 30), 'lon': (-40, -20)},   # North East Atlantic
    'SEP': {'lat': (-25, -5), 'lon': (-95, -75)},  # South East Pacific
    'NEP': {'lat': (15, 35), 'lon': (-135, -115)}, # North East Pacific
}
ACMCAP = ecio.load_ACMCAP(ACMCAP_file)
ACMCAP = ACMCAP.where((ACMCAP['latitude'] >= stratus_regions[region]['lat'][0]) & (ACMCAP['latitude'] <= stratus_regions[region]['lat'][1]), drop=True)

In [ ]:
ACMCAP.liquid_water_content.T.plot(figsize=(30,4), vmin=1e-5, vmax=1e-4, ylim=(240,200))

In [ ]:
i = 1
ACMCAP_i = ACMCAP.where((ACMCAP.latitude >= scene_midpoints['lat_min'][i]) & (ACMCAP.latitude <= scene_midpoints['lat_max'][i]), drop=True)

In [ ]:
import pandas as pd
i = 5
ACMCAP_i = ACMCAP.where((ACMCAP.time >= pd.to_datetime(scene_midpoints['time_mid'][i])) & (ACMCAP.time <= pd.to_datetime(scene_midpoints['time_mid'][i+1])), drop=True)

In [ ]:
ACMCAP_i.liquid_water_content.T.plot(figsize=(30,4), vmin=1e-5, vmax=1e-4, ylim=(240,200))

In [ ]:
ACMCAP_i

In [ ]:
for i in range(len(scene_midpoints['time_mid'])-1):
    ACMCAP_i = ACMCAP.where((ACMCAP.latitude >= scene_midpoints['lat_min'][i]) & (ACMCAP.latitude <= scene_midpoints['lat_max'][i]), drop=True)
    print(len(ACMCAP_i.along_track))
    ACMCAP_i = ACMCAP.where((ACMCAP.time >= pd.to_datetime(scene_midpoints['time_mid'][i])) & (ACMCAP.time <= pd.to_datetime(scene_midpoints['time_mid'][i+1])), drop=True)
    print(len(ACMCAP_i.along_track))
    print()